# DSA 4060 – Recommender Systems
## Week 3 Practical Lab: Building a Simple Content-Based Recommender

### Objective
The objective of this lab is to build a simple content-based movie recommender using structured movie features and user ratings. The system will create item profiles, build a user preference profile, calculate recommendation scores, remove already watched movies, and generate Top-N recommendations.

In [28]:
import pandas as pd

In [29]:
movies = pd.DataFrame({
    "movie_id": [1,2,3,4,5,6,7,8,9,10],
    "title": [
        "Galaxy Mission",
        "Love in Nairobi",
        "Final Battle",
        "Family Weekend",
        "Future World",
        "Broken Hearts",
        "City Detectives",
        "Space Warriors",
        "Wedding Weekend",
        "Hidden Truth"
    ],
    "Action":  [1,0,1,0,1,0,1,1,0,0],
    "Comedy":  [0,0,0,1,0,0,0,0,1,0],
    "Drama":   [0,1,0,0,0,1,1,0,0,1],
    "Romance": [0,1,0,0,0,1,0,0,1,0],
    "SciFi":   [1,0,0,0,1,0,0,1,0,0]
})

movies

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1
1,2,Love in Nairobi,0,0,1,1,0
2,3,Final Battle,1,0,0,0,0
3,4,Family Weekend,0,1,0,0,0
4,5,Future World,1,0,0,0,1
5,6,Broken Hearts,0,0,1,1,0
6,7,City Detectives,1,0,1,0,0
7,8,Space Warriors,1,0,0,0,1
8,9,Wedding Weekend,0,1,0,1,0
9,10,Hidden Truth,0,0,1,0,0


In [30]:
movies.to_csv("movies.csv", index=False)

In [31]:
movies.head()

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1
1,2,Love in Nairobi,0,0,1,1,0
2,3,Final Battle,1,0,0,0,0
3,4,Family Weekend,0,1,0,0,0
4,5,Future World,1,0,0,0,1


In [32]:
movies.shape

(10, 7)

In [33]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   movie_id  10 non-null     int64
 1   title     10 non-null     str  
 2   Action    10 non-null     int64
 3   Comedy    10 non-null     int64
 4   Drama     10 non-null     int64
 5   Romance   10 non-null     int64
 6   SciFi     10 non-null     int64
dtypes: int64(6), str(1)
memory usage: 828.0 bytes


### Dataset Exploration

1. **How many movies are in the dataset?**  
   There are 10 movies in the dataset.

2. **How many genre features exist?**  
   There are 5 genre features: Action, Comedy, Drama, Romance, and SciFi.

3. **What does one row represent?**  
   Each row represents one movie, including its unique movie ID, title, and binary genre features. A value of 1 indicates that the movie belongs to that genre, while 0 indicates that it does not.

In [34]:
features = ["Action", "Comedy", "Drama", "Romance", "SciFi"]

movie_features = movies[features]

movie_features

,Action,Comedy,Drama,Romance,SciFi
0,1,0,0,0,1
1,0,0,1,1,0
2,1,0,0,0,0
3,0,1,0,0,0
4,1,0,0,0,1
5,0,0,1,1,0
6,1,0,1,0,0
7,1,0,0,0,1
8,0,1,0,1,0
9,0,0,1,0,0


In [35]:
movies.loc[
    movies["title"] == "Galaxy Mission"
]

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1


### Item Profile Interpretation

The 1s indicate genres that are present in the movie, while the 0s indicate genres that are absent. Galaxy Mission has Action = 1 and SciFi = 1, meaning it is an Action and SciFi movie.

In [36]:
ratings = pd.DataFrame({
    "movie_id": [1, 2, 3, 5],
    "rating": [5, 2, 4, 5]
})

ratings

,movie_id,rating
0,1,5
1,2,2
2,3,4
3,5,5


In [37]:
user_movies = ratings.merge(
    movies,
    on="movie_id"
)

user_movies

,movie_id,rating,title,Action,Comedy,Drama,Romance,SciFi
0,1,5,Galaxy Mission,1,0,0,0,1
1,2,2,Love in Nairobi,0,0,1,1,0
2,3,4,Final Battle,1,0,0,0,0
3,5,5,Future World,1,0,0,0,1


In [38]:
user_movies[
    ["title", "rating", "Action", "Comedy", "Drama", "Romance", "SciFi"]
]

,title,rating,Action,Comedy,Drama,Romance,SciFi
0,Galaxy Mission,5,1,0,0,0,1
1,Love in Nairobi,2,0,0,1,1,0
2,Final Battle,4,1,0,0,0,0
3,Future World,5,1,0,0,0,1


### User Preference Observation

User A appears to prefer Action and SciFi movies. Galaxy Mission and Future World both received 5-star ratings, while Final Battle, an Action movie, received 4 stars.

In [39]:
rated_features = user_movies[features]
user_ratings = user_movies["rating"]

rated_features

,Action,Comedy,Drama,Romance,SciFi
0,1,0,0,0,1
1,0,0,1,1,0
2,1,0,0,0,0
3,1,0,0,0,1


In [40]:
user_ratings

0    5
1    2
2    4
3    5
Name: rating, dtype: int64

In [41]:
weighted_features = rated_features.mul(
    user_ratings,
    axis=0
)

weighted_features

,Action,Comedy,Drama,Romance,SciFi
0,5,0,0,0,5
1,0,0,2,2,0
2,4,0,0,0,0
3,5,0,0,0,5


In [42]:
user_profile = weighted_features.sum()

user_profile

Action     14
Comedy      0
Drama       2
Romance     2
SciFi      10
dtype: int64

In [43]:
user_profile.sort_values(
    ascending=False
)

Action     14
SciFi      10
Drama       2
Romance     2
Comedy      0
dtype: int64

### User Profile Interpretation

The user's two strongest genre preferences are Action and SciFi. Action has the highest profile score of 14, followed by SciFi with a score of 10.

In [44]:
user_profile_normalized = (
    user_profile / user_profile.sum()
)

user_profile_normalized

Action     0.500000
Comedy     0.000000
Drama      0.071429
Romance    0.071429
SciFi      0.357143
dtype: float64

In [45]:
movies["recommendation_score"] = (
    movie_features
    .mul(user_profile_normalized)
    .sum(axis=1)
)

movies[["title", "recommendation_score"]]

,title,recommendation_score
0,Galaxy Mission,0.857143
1,Love in Nairobi,0.142857
2,Final Battle,0.500000
3,Family Weekend,0.000000
4,Future World,0.857143
5,Broken Hearts,0.142857
6,City Detectives,0.571429
7,Space Warriors,0.857143
8,Wedding Weekend,0.071429
9,Hidden Truth,0.071429


In [46]:
ranked_movies = movies.sort_values(
    "recommendation_score",
    ascending=False
)

ranked_movies[["title", "recommendation_score"]]

,title,recommendation_score
0,Galaxy Mission,0.857143
4,Future World,0.857143
7,Space Warriors,0.857143
6,City Detectives,0.571429
2,Final Battle,0.500000
1,Love in Nairobi,0.142857
5,Broken Hearts,0.142857
8,Wedding Weekend,0.071429
9,Hidden Truth,0.071429
3,Family Weekend,0.000000


In [47]:
watched_movies = ratings["movie_id"]

recommendations = ranked_movies[
    ~ranked_movies["movie_id"].isin(watched_movies)
]

recommendations[["title", "recommendation_score"]]

,title,recommendation_score
7,Space Warriors,0.857143
6,City Detectives,0.571429
5,Broken Hearts,0.142857
8,Wedding Weekend,0.071429
9,Hidden Truth,0.071429
3,Family Weekend,0.000000


In [48]:
top_3 = recommendations[
    ["title", "recommendation_score"]
].head(3)

top_3

,title,recommendation_score
7,Space Warriors,0.857143
6,City Detectives,0.571429
5,Broken Hearts,0.142857


### Top 3 Recommendation Explanation

1. **Space Warriors** ranks first because it contains both Action and SciFi, which are User A's two strongest preferences.
2. **City Detectives** ranks second because it contains Action, the user's strongest genre preference.
3. **Hidden Truth** ranks third because it contains Drama, which has a smaller positive weight in the user's profile.

In [49]:
def recommend_movies(movies, ratings, features, n=3):
    user_movies = ratings.merge(movies, on="movie_id")

    rated_features = user_movies[features]

    weighted_features = rated_features.mul(
        user_movies["rating"],
        axis=0
    )

    user_profile = weighted_features.sum()
    user_profile = user_profile / user_profile.sum()

    scores = (
        movies[features]
        .mul(user_profile)
        .sum(axis=1)
    )

    results = movies.copy()
    results["recommendation_score"] = scores

    watched = ratings["movie_id"]

    results = results[
        ~results["movie_id"].isin(watched)
    ]

    return results.sort_values(
        "recommendation_score",
        ascending=False
    ).head(n)

In [50]:
recommend_movies(
    movies,
    ratings,
    features
)

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
7,8,Space Warriors,1,0,0,0,1,0.857143
6,7,City Detectives,1,0,1,0,0,0.571429
5,6,Broken Hearts,0,0,1,1,0,0.142857


In [51]:
user_b_ratings = pd.DataFrame({
    "movie_id": [2, 4, 6, 9],
    "rating": [5, 4, 5, 4]
})

recommend_movies(
    movies,
    user_b_ratings,
    features
)

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
6,7,City Detectives,1,0,1,0,0,0.3125
9,10,Hidden Truth,0,0,1,0,0,0.3125
0,1,Galaxy Mission,1,0,0,0,1,0.0000


### User A vs User B

- User A and User B do not receive the same recommendations because their ratings produce different preference profiles. 
- User A has stronger preferences for Action and SciFi, while User B's ratings give more importance to Drama and Romance.
-  Therefore, the recommender ranks movies differently for each user.

### User C – Thinking Exercise

1. **What will the model learn about User C?**  
   The model will learn that User C appears to prefer the features of Galaxy Mission.

2. **Which genres will dominate the profile?**  
   Action and SciFi will dominate because Galaxy Mission belongs to both genres.

3. **What could go wrong with the recommendations?**  
   The recommendations may become too narrow because the model assumes that Action and SciFi represent the user's overall preferences.

4. **Is one rating enough to understand a user's preferences?**  
   No. One rating provides very limited information and may not accurately represent the user's overall movie preferences.

In [52]:
new_movie = {
    "movie_id": 11,
    "title": "Nairobi After Dark",
    "Action": 1,
    "Comedy": 0,
    "Drama": 1,
    "Romance": 0,
    "SciFi": 1
}

movies = pd.concat(
    [movies, pd.DataFrame([new_movie])],
    ignore_index=True
)

movies.tail()

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
6,7,City Detectives,1,0,1,0,0,0.571429
7,8,Space Warriors,1,0,0,0,1,0.857143
8,9,Wedding Weekend,0,1,0,1,0,0.071429
9,10,Hidden Truth,0,0,1,0,0,0.071429
10,11,Nairobi After Dark,1,0,1,0,1,NaN


In [53]:
movies.to_csv("movies.csv", index=False)

In [54]:
my_user = pd.DataFrame({
    "movie_id": [1, 2, 6, 8, 10],
    "rating": [5, 2, 3, 5, 2]
})

my_top_5 = recommend_movies(
    movies,
    my_user,
    features,
    n=5
)

my_top_5

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
10,11,Nairobi After Dark,1,0,1,0,1,0.84375
4,5,Future World,1,0,0,0,1,0.62500
6,7,City Detectives,1,0,1,0,0,0.53125
2,3,Final Battle,1,0,0,0,0,0.31250
8,9,Wedding Weekend,0,1,0,1,0,0.15625


### Interpretation of My Recommendations

1. **What are my user's two strongest preferences?**  
   The user's two strongest preferences are Action and SciFi.

2. **Which three movies received the highest recommendation scores?**  
   The three highest-ranked movies were Nairobi After Dark (0.84375), Future World (0.62500), and City Detectives (0.53125).

3. **Why did the top movie rank first?**  
   Nairobi After Dark ranked first because it contains Action, Drama, and SciFi features, giving it the strongest overall match with the user's preference profile.

4. **Did any recommendation surprise me?**  
   Wedding Weekend was somewhat surprising because it does not contain Action or SciFi, which are the user's strongest preferences. However, it still received a lower recommendation score because it matches some of the user's other preferences.

5. **What important information is the model ignoring?**  
   The model ignores important information such as actors, directors, release year, language, movie descriptions, popularity, and other contextual information.

### Limitations

1. The recommender uses only five genre features, so it has a limited understanding of each movie.

2. It ignores useful information such as actors, directors, release year, language, and movie descriptions.

3. New users with very few ratings may receive inaccurate recommendations because the system has insufficient information about their preferences.

4. The system may recommend movies that are too similar to movies the user has already liked.

### Content-Based Filtering Workflow

Item Metadata → Item Profiles → User Ratings → User Profile → Candidate Scores → Remove Seen Items → Rank Items → Top-N Recommendations

### Conclusion

- In this lab, a simple content-based movie recommender was developed using structured genre features and user ratings. 
- Movie features were used to construct item profiles, while ratings were used to build user preference profiles. 
- The system then calculated recommendation scores, removed previously watched movies, and generated Top-N recommendations.
-  The results demonstrate how different user preferences can produce personalized recommendations.

### Final Question

- No. Two users who watched the same movies but gave them different ratings should not necessarily receive the same recommendations. - In this model, ratings are used to weight the movie features when constructing the user profile. 
- Therefore, different ratings can produce different feature weights and different user preference profiles.
- As a result, the recommendation scores and rankings can also be different.